# Fine-tuning medical LoRA - TechCorp\n\nNotebook Colab court pour produire des metriques reelles sur `ruslanmv/ai-medical-chatbot`. Usage experimental uniquement.

In [ ]:
!pip install -q transformers==4.45.2 datasets peft accelerate bitsandbytes

In [ ]:
from datasets import load_dataset\n\nDATASET = 'ruslanmv/ai-medical-chatbot'\nraw = load_dataset(DATASET, split='train')\nraw = raw.shuffle(seed=42).select(range(1200))\n\ndef format_row(row):\n    desc = (row.get('Description') or '').strip()\n    patient = (row.get('Patient') or '').strip()\n    doctor = (row.get('Doctor') or '').strip()\n    prompt = 'Tu es un assistant medical experimental prudent. Rappelle que cela ne remplace pas un avis medical professionnel.\\n'\n    if desc:\n        prompt += f'Contexte: {desc}\\n'\n    prompt += f'Patient: {patient}\\nAssistant:'\n    return {'text': prompt + ' ' + doctor}\n\ndataset = raw.map(format_row, remove_columns=raw.column_names)\nsplit = dataset.train_test_split(test_size=0.1, seed=42)\ntrain_ds = split['train']\neval_ds = split['test']\nlen(train_ds), len(eval_ds), train_ds[0]['text'][:300]

In [ ]:
import torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\nfrom peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training\n\nBASE_MODEL = 'microsoft/Phi-3.5-mini-instruct'\ntokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)\nif tokenizer.pad_token is None:\n    tokenizer.pad_token = tokenizer.eos_token\n\nbnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)\nmodel = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map='auto', trust_remote_code=True)\nmodel = prepare_model_for_kbit_training(model)\nlora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['qkv_proj', 'o_proj', 'gate_up_proj', 'down_proj'])\nmodel = get_peft_model(model, lora_config)\nmodel.print_trainable_parameters()

In [ ]:
def tokenize(batch):\n    tokens = tokenizer(batch['text'], truncation=True, padding='max_length', max_length=512)\n    tokens['labels'] = tokens['input_ids'].copy()\n    return tokens\n\ntrain_tok = train_ds.map(tokenize, batched=True, remove_columns=['text'])\neval_tok = eval_ds.map(tokenize, batched=True, remove_columns=['text'])

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling\n\nargs = TrainingArguments(\n    output_dir='medical_lora_phi35',\n    max_steps=30,\n    per_device_train_batch_size=1,\n    gradient_accumulation_steps=4,\n    learning_rate=2e-4,\n    logging_steps=5,\n    eval_strategy='steps',\n    eval_steps=10,\n    save_steps=30,\n    fp16=True,\n    report_to='none'\n)\ncollator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)\ntrainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=eval_tok, data_collator=collator)\ntrain_result = trainer.train()\neval_result = trainer.evaluate()\ntrainer.save_model('medical_lora_phi35/final_adapter')\nprint({'train_loss': train_result.training_loss, 'eval_loss': eval_result.get('eval_loss'), 'max_steps': args.max_steps})